# PASUL 4 — Generare TFRecords cu offline preprocessing

**Scop:** Pentru fiecare imagine din pipeline (val_fixed, test_fixed, tier1-5):
1. Citim imaginea raw de pe Drive
2. Aplicăm preprocessing-ul OFFLINE (validat empiric în PASUL 3 explorer):
   - Black border crop (heuristic)
   - Shades of Gray color constancy (p=6)
   - Resize la 256×256 (aspect-ratio-preserving + center crop)
   - Encode JPEG quality 95
3. Salvăm într-un TFRecord (un fișier per split, sharded)

**Output:**
```
pipeline_outputs/step4_tfrecords/
├── val/
│   ├── val-00000-of-00003.tfrecord
│   ├── val-00001-of-00003.tfrecord
│   └── val-00002-of-00003.tfrecord
├── test/
│   └── ...
├── tier1/
│   └── ...
├── tier2/
│   └── ...
...
└── tier5/
    └── ...
```

**Decizii cheie:**

- **Sharding** (mai multe fișiere mici per split): permite paralelizare + recovery dacă crapă Colab
- **Skip dacă există**: dacă un shard e deja generat, sărim peste — permite resume după întrerupere
- **Procesare paralelă** cu `multiprocessing` pentru CPU augment (NU pe Drive direct, descarcăm batch-uri în RAM)
- **Salvare progresivă**: scriem fiecare shard imediat după ce-l procesăm, ca să nu pierdem progres

**Estimare timp:**
- ~25k imagini × 3 splituri (val + test + tiers cumulative)
- T1 e subset al T5, deci nu reprocesăm aceleași imagini de 5 ori — generăm tier5 + val + test, iar T1-T4 sunt subseturi (filtrate la antrenare)
- Total ~25k imagini × ~80ms preprocessing = ~33 minute pe CPU + I/O Drive (~1.5h total)

---

## STRATEGIE: Generăm DOAR 3 seturi TFRecord

Pentru eficiență, NU generăm 7 TFRecord-uri separate (val + test + tier1..5). În loc, observăm că:

- **val_fixed** = 1 TFRecord (folosit pentru toate experimentele)
- **test_fixed** = 1 TFRecord (folosit DOAR la final, pe model best)
- **tier5_train** = 1 TFRecord (cuprinde tot train_pool, deci T1-T4 sunt subseturi)

La antrenare, pentru tier_N, **citim tier5 TFRecord și filtrăm in-memory** după lesion_id-urile din tier_N CSV. Asta:
- Economisește 4× generare TFRecord (de la 5 la 1 pentru train)
- Garantează că tier_N e exact subset al tier5 (cumulativitate)
- Permite schimbarea filtrelor fără regenerare TFRecord

Vom genera deci **3 TFRecord-uri**: `val/`, `test/`, `tier5/` (= train_pool complet preprocesat).

---

## 1. Setup

In [ ]:
# Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Install opencv (în Colab e deja instalat)
import subprocess
try:
    import cv2
except ImportError:
    subprocess.check_call(['pip', 'install', 'opencv-python-headless', '-q'])
    import cv2

import os
import io
import time
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from tqdm import tqdm

print(f"TensorFlow: {tf.__version__}")
print(f"OpenCV: {cv2.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU'))}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow: 2.20.0
OpenCV: 4.13.0
GPU available: 1


In [ ]:
# === CONFIGURARE ===

BASE_DIR = '/content/drive/MyDrive/Thesis'

# Input din pașii anteriori
VAL_CSV   = f'{BASE_DIR}/pipeline_outputs/step1/val_fixed.csv'
TEST_CSV  = f'{BASE_DIR}/pipeline_outputs/step1/test_fixed.csv'
TIER5_CSV = f'{BASE_DIR}/pipeline_outputs/step2/tier5_train.csv'

# Output
OUTPUT_DIR = f'{BASE_DIR}/pipeline_outputs/step4_tfrecords'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parametri preprocessing (validate empiric în PASUL 3)
TARGET_SIZE = 256              # imagini salvate la 256×256
JPEG_QUALITY = 95              # păstrare detalii
SOG_POWER = 6                  # Shades of Gray Minkowski norm
BLACK_THRESHOLD = 10           # pixel intensity sub care = "negru"

# Sharding (împărțim fiecare split în multiple fișiere)
SHARD_SIZE = 1000              # imagini per shard

# Mapare clase
LABEL_TO_IDX = {
    'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3,
    'mel': 4, 'nv': 5, 'vasc': 6
}

print("Configurare:")
print(f"  TARGET_SIZE: {TARGET_SIZE}×{TARGET_SIZE}")
print(f"  JPEG_QUALITY: {JPEG_QUALITY}")
print(f"  SHARD_SIZE: {SHARD_SIZE}")
print(f"  Output: {OUTPUT_DIR}")

Configurare:
  TARGET_SIZE: 256×256
  JPEG_QUALITY: 95
  SHARD_SIZE: 1000
  Output: /content/drive/MyDrive/Thesis/pipeline_outputs/step4_tfrecords


## 2. Funcții preprocessing offline

Aceleași funcții validate în PASUL 3 (preprocessing_explorer).

In [ ]:
def crop_black_border(img, threshold=BLACK_THRESHOLD):
    """Crop margine neagră (Gessert et al. 2020)."""
    gray = np.mean(img, axis=2)
    mask = gray > threshold
    if not mask.any():
        return img
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return img[rmin:rmax+1, cmin:cmax+1]


def shades_of_gray(img, power=SOG_POWER):
    """Color constancy (Finlayson & Trezzi 2004)."""
    img_float = img.astype(np.float32)
    img_power = np.power(img_float, power)
    rgb_vec = np.power(np.mean(img_power, axis=(0, 1)), 1/power)
    rgb_norm = np.sqrt(np.sum(rgb_vec ** 2))
    if rgb_norm == 0:
        return img
    rgb_vec = rgb_vec / rgb_norm
    rgb_vec = 1 / (rgb_vec * np.sqrt(3))
    img_corrected = img_float * rgb_vec
    return np.clip(img_corrected, 0, 255).astype(np.uint8)


def resize_with_aspect(img, target=TARGET_SIZE):
    """Resize cu păstrare aspect ratio: latura mai scurtă → target."""
    h, w = img.shape[:2]
    if h < w:
        new_h = target
        new_w = int(w * target / h)
    else:
        new_w = target
        new_h = int(h * target / w)
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)


def center_crop(img, size=TARGET_SIZE):
    """Center crop la size×size."""
    h, w = img.shape[:2]
    start_h = max(0, (h - size) // 2)
    start_w = max(0, (w - size) // 2)
    return img[start_h:start_h+size, start_w:start_w+size]


def encode_jpeg(img, quality=JPEG_QUALITY):
    """Encode imagine ca bytes JPEG."""
    # cv2 lucrează în BGR, convertim
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    success, encoded = cv2.imencode('.jpg', img_bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    if not success:
        raise RuntimeError("JPEG encoding failed")
    return encoded.tobytes()


def preprocess_image(image_path):
    """Pipeline complet offline."""
    # 1. Citire (PIL pentru robustețe vs cv2 cu nume non-ASCII)
    img = np.array(Image.open(image_path).convert('RGB'))

    # 2. Black border crop
    img = crop_black_border(img)

    # 3. Shades of Gray
    img = shades_of_gray(img)

    # 4. Resize aspect-preserving
    img = resize_with_aspect(img)

    # 5. Center crop
    img = center_crop(img)

    # 6. Encode JPEG
    return encode_jpeg(img)


print("Funcții preprocessing definite.")

Funcții preprocessing definite.


## 3. Test rapid pipeline pe o imagine

Verificăm că funcțiile lucrează corect înainte de a procesa 25k imagini.

In [ ]:
# Test pe prima imagine din val_fixed
val_df = pd.read_csv(VAL_CSV)
test_path = val_df.iloc[0]['image_path']

print(f"Test pe: {test_path}")
print(f"Există: {os.path.exists(test_path)}")

if os.path.exists(test_path):
    t0 = time.time()
    jpeg_bytes = preprocess_image(test_path)
    elapsed_ms = (time.time() - t0) * 1000

    print(f"Procesare timp: {elapsed_ms:.1f} ms")
    print(f"JPEG bytes: {len(jpeg_bytes):,} ({len(jpeg_bytes)/1024:.1f} KB)")

    # Decode înapoi pentru verificare
    decoded = tf.io.decode_jpeg(jpeg_bytes).numpy()
    print(f"Shape după decode: {decoded.shape}")
    print(f"Dtype: {decoded.dtype}, range: [{decoded.min()}, {decoded.max()}]")

    # Vizualizare quick
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(np.array(Image.open(test_path).convert('RGB')))
    axes[0].set_title(f"RAW")
    axes[0].axis('off')
    axes[1].imshow(decoded)
    axes[1].set_title(f"DUPĂ pipeline (256×256)")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("[EROARE] Path-ul nu există!")

## 4. TFRecord writer cu sharding și resume

Scriem TFRecord-urile în shard-uri de câte 1000 imagini. Dacă un shard există deja, îl sărim — permite resume după întrerupere.

In [ ]:
def make_example(image_id, label_idx, lesion_id, source, jpeg_bytes):
    """Construiește un tf.train.Example pentru o imagine."""
    feature = {
        'image_id': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[image_id.encode('utf-8')])
        ),
        'label': tf.train.Feature(
            int64_list=tf.train.Int64List(value=[label_idx])
        ),
        'lesion_id': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[lesion_id.encode('utf-8')])
        ),
        'source': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[source.encode('utf-8')])
        ),
        'image_bytes': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[jpeg_bytes])
        ),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature))


def write_split_to_tfrecord(df, split_name, output_dir, shard_size=SHARD_SIZE):
    """
    Scrie un dataframe ca TFRecord-uri cu sharding și resume.

    Args:
        df: DataFrame cu coloanele image_id, image_path, dx, lesion_id, source
        split_name: 'val', 'test', sau 'tier5'
        output_dir: directorul output
        shard_size: numărul de imagini per shard
    """
    split_dir = os.path.join(output_dir, split_name)
    os.makedirs(split_dir, exist_ok=True)

    n_total = len(df)
    n_shards = (n_total + shard_size - 1) // shard_size

    print(f"\n=== Scriere {split_name}: {n_total:,} imagini → {n_shards} shard-uri ===")

    # Statistici globale
    stats = {
        'total': n_total,
        'processed': 0,
        'skipped_existing': 0,
        'failed': [],
        'avg_ms_per_image': 0
    }

    times = []

    for shard_idx in range(n_shards):
        shard_path = os.path.join(
            split_dir,
            f'{split_name}-{shard_idx:05d}-of-{n_shards:05d}.tfrecord'
        )

        # Skip dacă shard-ul există deja
        if os.path.exists(shard_path):
            shard_size_actual = shard_size if shard_idx < n_shards - 1 else (n_total - shard_idx * shard_size)
            stats['skipped_existing'] += shard_size_actual
            print(f"  [SKIP] Shard {shard_idx+1}/{n_shards} există deja")
            continue

        # Imaginile din shard-ul curent
        start = shard_idx * shard_size
        end = min(start + shard_size, n_total)
        shard_df = df.iloc[start:end]

        # Scriem temporar într-un fișier .tmp, apoi rename atomic
        tmp_path = shard_path + '.tmp'

        with tf.io.TFRecordWriter(tmp_path) as writer:
            for _, row in tqdm(shard_df.iterrows(),
                              total=len(shard_df),
                              desc=f"  Shard {shard_idx+1}/{n_shards}",
                              leave=False):
                try:
                    t0 = time.time()
                    jpeg_bytes = preprocess_image(row['image_path'])
                    times.append(time.time() - t0)

                    label_idx = LABEL_TO_IDX[row['dx']]
                    example = make_example(
                        image_id=row['image_id'],
                        label_idx=label_idx,
                        lesion_id=row['lesion_id'],
                        source=row['source'],
                        jpeg_bytes=jpeg_bytes
                    )
                    writer.write(example.SerializeToString())
                    stats['processed'] += 1
                except Exception as e:
                    stats['failed'].append((row['image_id'], str(e)))
                    print(f"\n  [FAIL] {row['image_id']}: {e}")

        # Rename atomic .tmp -> .tfrecord
        os.rename(tmp_path, shard_path)

        if times:
            stats['avg_ms_per_image'] = np.mean(times) * 1000

    return stats


print("TFRecord writer definit.")

TFRecord writer definit.


## 5. Generare val_fixed și test_fixed (mici, rapide)

Rulăm întâi seturile mici (~2.5k imagini fiecare). Asta e un test bun: dacă ceva e greșit cu pipeline-ul, descoperim repede pe puține imagini.

In [ ]:
# Citire CSV-uri
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"val_fixed: {len(val_df):,} imagini")
print(f"test_fixed: {len(test_df):,} imagini")
print(f"\nDistribuție clase val:")
print(val_df['dx'].value_counts().to_string())

val_fixed: 2,519 imagini
test_fixed: 2,513 imagini

Distribuție clase val:
dx
nv       1273
mel       466
bcc       334
bkl       250
akiec     149
df         24
vasc       23


In [ ]:
# Generare val
t0 = time.time()
val_stats = write_split_to_tfrecord(val_df, 'val', OUTPUT_DIR)
val_time = time.time() - t0

print(f"\n=== VAL TERMINAT ===")
print(f"  Procesate: {val_stats['processed']:,}/{val_stats['total']:,}")
print(f"  Skipped existing: {val_stats['skipped_existing']:,}")
print(f"  Failed: {len(val_stats['failed'])}")
print(f"  Timp: {val_time/60:.1f} min")
print(f"  Avg per imagine: {val_stats['avg_ms_per_image']:.1f} ms")


=== Scriere val: 2,519 imagini → 3 shard-uri ===



=== VAL TERMINAT ===
  Procesate: 2,519/2,519
  Skipped existing: 0
  Failed: 0
  Timp: 32.3 min
  Avg per imagine: 768.7 ms


In [ ]:
# Generare test
t0 = time.time()
test_stats = write_split_to_tfrecord(test_df, 'test', OUTPUT_DIR)
test_time = time.time() - t0

print(f"\n=== TEST TERMINAT ===")
print(f"  Procesate: {test_stats['processed']:,}/{test_stats['total']:,}")
print(f"  Skipped existing: {test_stats['skipped_existing']:,}")
print(f"  Failed: {len(test_stats['failed'])}")
print(f"  Timp: {test_time/60:.1f} min")
print(f"  Avg per imagine: {test_stats['avg_ms_per_image']:.1f} ms")


=== Scriere test: 2,513 imagini → 3 shard-uri ===



=== TEST TERMINAT ===
  Procesate: 2,513/2,513
  Skipped existing: 0
  Failed: 0
  Timp: 32.4 min
  Avg per imagine: 772.9 ms


## 6. Verificare TFRecord-uri val/test

Citim ce am scris și verificăm că totul e corect.

In [ ]:
def parse_tfrecord(example_proto):
    feature_desc = {
        'image_id': tf.io.FixedLenFeature([], tf.string),
        'label': tf.io.FixedLenFeature([], tf.int64),
        'lesion_id': tf.io.FixedLenFeature([], tf.string),
        'source': tf.io.FixedLenFeature([], tf.string),
        'image_bytes': tf.io.FixedLenFeature([], tf.string),
    }
    return tf.io.parse_single_example(example_proto, feature_desc)


def verify_split(split_name, expected_count):
    """Citește toate shard-urile dintr-un split și verifică numărătoarea."""
    split_dir = os.path.join(OUTPUT_DIR, split_name)
    files = sorted(tf.io.gfile.glob(f'{split_dir}/*.tfrecord'))

    print(f"\n=== Verificare {split_name} ===")
    print(f"  Shard-uri găsite: {len(files)}")

    ds = tf.data.TFRecordDataset(files).map(parse_tfrecord)

    count = 0
    label_dist = {}
    source_dist = {}
    sample = None

    for ex in ds:
        count += 1
        label = int(ex['label'].numpy())
        source = ex['source'].numpy().decode()
        label_dist[label] = label_dist.get(label, 0) + 1
        source_dist[source] = source_dist.get(source, 0) + 1
        if sample is None:
            sample = ex

    print(f"  Records: {count:,} (expected: {expected_count:,})")

    if count != expected_count:
        print(f"  [WARNING] Numărătoare diferită!")
    else:
        print(f"  [OK] Numărătoare corectă")

    print(f"\n  Distribuție clase (label idx → count):")
    for k in sorted(label_dist.keys()):
        idx_to_label = {v: k for k, v in LABEL_TO_IDX.items()}
        print(f"    {k} ({idx_to_label[k]}): {label_dist[k]:,}")

    print(f"\n  Distribuție surse:")
    for k, v in sorted(source_dist.items()):
        print(f"    {k}: {v:,}")

    # Decode sample
    if sample is not None:
        img = tf.io.decode_jpeg(sample['image_bytes'])
        print(f"\n  Sample image:")
        print(f"    image_id: {sample['image_id'].numpy().decode()}")
        print(f"    shape: {img.shape}")
        print(f"    range: [{tf.reduce_min(img).numpy()}, {tf.reduce_max(img).numpy()}]")

    return count


val_count = verify_split('val', len(val_df))
test_count = verify_split('test', len(test_df))

## 7. Generare tier5 (training pool complet)

Asta e cea mai mare parte: ~20,300 imagini. Va dura ~30-50 minute.

**Important:** dacă Colab te deconectează, **rerularea celulei va continua de unde a rămas** datorită mecanismului de skip pe shard-uri existente. Deci nu ai de ce să-ți faci griji.

In [ ]:
tier5_df = pd.read_csv(TIER5_CSV)
print(f"tier5: {len(tier5_df):,} imagini")
print(f"\nDistribuție clase:")
print(tier5_df['dx'].value_counts().to_string())

tier5: 20,299 imagini

Distribuție clase:
dx
nv       10297
mel       3628
bcc       2661
bkl       2128
akiec     1192
vasc       209
df         184


In [ ]:
# Generare tier5 — CEL MAI LUNG PAS
t0 = time.time()
tier5_stats = write_split_to_tfrecord(tier5_df, 'tier5', OUTPUT_DIR)
tier5_time = time.time() - t0

print(f"\n=== TIER5 TERMINAT ===")
print(f"  Procesate: {tier5_stats['processed']:,}/{tier5_stats['total']:,}")
print(f"  Skipped existing: {tier5_stats['skipped_existing']:,}")
print(f"  Failed: {len(tier5_stats['failed'])}")
print(f"  Timp: {tier5_time/60:.1f} min")
if tier5_stats['avg_ms_per_image'] > 0:
    print(f"  Avg per imagine: {tier5_stats['avg_ms_per_image']:.1f} ms")


=== Scriere tier5: 20,299 imagini → 21 shard-uri ===



=== TIER5 TERMINAT ===
  Procesate: 20,299/20,299
  Skipped existing: 0
  Failed: 0
  Timp: 257.3 min
  Avg per imagine: 759.0 ms


In [ ]:
# Verificare tier5
tier5_count = verify_split('tier5', len(tier5_df))

## 8. Statistici disk usage

In [ ]:
def get_dir_size(path):
    """Returnează mărimea totală în MB."""
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / (1024 ** 2)


print("Disk usage:")
for split in ['val', 'test', 'tier5']:
    split_dir = os.path.join(OUTPUT_DIR, split)
    if os.path.exists(split_dir):
        size_mb = get_dir_size(split_dir)
        n_shards = len(tf.io.gfile.glob(f'{split_dir}/*.tfrecord'))
        print(f"  {split:6s}: {size_mb:>7.1f} MB ({n_shards} shard-uri)")

total_mb = get_dir_size(OUTPUT_DIR)
print(f"\nTotal: {total_mb:.1f} MB ({total_mb/1024:.2f} GB)")

## 9. Raport eșuări (dacă există)

In [ ]:
all_failures = (
    val_stats['failed'] +
    test_stats['failed'] +
    tier5_stats['failed']
)

if not all_failures:
    print("[OK] Zero imagini au eșuat la procesare.")
else:
    print(f"[ATENȚIE] {len(all_failures)} imagini au eșuat:")
    for image_id, error in all_failures[:20]:
        print(f"  {image_id}: {error}")

    # Salvează lista de eșuări
    failures_path = os.path.join(OUTPUT_DIR, 'failures.json')
    with open(failures_path, 'w') as f:
        json.dump(all_failures, f, indent=2)
    print(f"\nSalvat: {failures_path}")

## 10. Metadata pentru pașii următori

Salvăm un fișier JSON cu informații pentru PASUL 5 (training).

In [ ]:
metadata = {
    'preprocessing_config': {
        'target_size': TARGET_SIZE,
        'jpeg_quality': JPEG_QUALITY,
        'shades_of_gray_power': SOG_POWER,
        'black_threshold': BLACK_THRESHOLD,
        'pipeline': [
            '1. crop_black_border (Gessert 2020)',
            '2. shades_of_gray p=6 (Finlayson & Trezzi 2004)',
            '3. resize_with_aspect (shorter side -> target_size)',
            '4. center_crop (target_size x target_size)',
            '5. JPEG encode quality=95'
        ]
    },
    'splits': {
        'val': {
            'count': val_count,
            'shards': len(tf.io.gfile.glob(f'{OUTPUT_DIR}/val/*.tfrecord')),
            'csv_source': VAL_CSV
        },
        'test': {
            'count': test_count,
            'shards': len(tf.io.gfile.glob(f'{OUTPUT_DIR}/test/*.tfrecord')),
            'csv_source': TEST_CSV
        },
        'tier5': {
            'count': tier5_count,
            'shards': len(tf.io.gfile.glob(f'{OUTPUT_DIR}/tier5/*.tfrecord')),
            'csv_source': TIER5_CSV,
            'note': 'tier1-4 are subseturi filtrate la antrenare prin lesion_id'
        }
    },
    'label_to_idx': LABEL_TO_IDX,
    'tfrecord_schema': {
        'image_id': 'tf.string',
        'label': 'tf.int64',
        'lesion_id': 'tf.string',
        'source': 'tf.string',
        'image_bytes': 'tf.string (JPEG bytes)'
    }
}

metadata_path = os.path.join(OUTPUT_DIR, 'metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"[SAVED] {metadata_path}")
print(json.dumps(metadata, indent=2))

[SAVED] /content/drive/MyDrive/Thesis/pipeline_outputs/step4_tfrecords/metadata.json
{
  "preprocessing_config": {
    "target_size": 256,
    "jpeg_quality": 95,
    "shades_of_gray_power": 6,
    "black_threshold": 10,
    "pipeline": [
      "1. crop_black_border (Gessert 2020)",
      "2. shades_of_gray p=6 (Finlayson & Trezzi 2004)",
      "3. resize_with_aspect (shorter side -> target_size)",
      "4. center_crop (target_size x target_size)",
      "5. JPEG encode quality=95"
    ]
  },
  "splits": {
    "val": {
      "count": 2519,
      "shards": 3,
      "csv_source": "/content/drive/MyDrive/Thesis/pipeline_outputs/step1/val_fixed.csv"
    },
    "test": {
      "count": 2513,
      "shards": 3,
      "csv_source": "/content/drive/MyDrive/Thesis/pipeline_outputs/step1/test_fixed.csv"
    },
    "tier5": {
      "count": 20299,
      "shards": 21,
      "csv_source": "/content/drive/MyDrive/Thesis/pipeline_outputs/step2/tier5_train.csv",
      "note": "tier1-4 are subseturi f

## 11. Sample readback — verifică imagini efective

In [ ]:
# Vizualizează 8 imagini random din tier5
import matplotlib.pyplot as plt

tier5_files = sorted(tf.io.gfile.glob(f'{OUTPUT_DIR}/tier5/*.tfrecord'))
ds = tf.data.TFRecordDataset(tier5_files).map(parse_tfrecord).shuffle(1000, seed=42).take(8)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

idx_to_label = {v: k for k, v in LABEL_TO_IDX.items()}

for ax, ex in zip(axes, ds):
    img = tf.io.decode_jpeg(ex['image_bytes']).numpy()
    label = idx_to_label[int(ex['label'].numpy())]
    source = ex['source'].numpy().decode()
    image_id = ex['image_id'].numpy().decode()

    ax.imshow(img)
    ax.set_title(f"{image_id}\n{label} | {source}", fontsize=8)
    ax.axis('off')

plt.suptitle('Sample din tier5 TFRecord (8 imagini random)', fontsize=12)
plt.tight_layout()
plt.show()

## 12. Checklist final

Verifică înainte de a trece la PASUL 5:

- [ ] **val** TFRecord generat cu numărătoare corectă (~2,519 records)
- [ ] **test** TFRecord generat (~2,513 records)
- [ ] **tier5** TFRecord generat (~20,299 records)
- [ ] Zero eșuări la procesare (sau lista eșuărilor e <1%)
- [ ] Disk usage rezonabil (~3-5 GB total)
- [ ] Sample readback arată imagini corecte (256×256, color-correct)
- [ ] `metadata.json` salvat

---

## Conexiune cu PASUL 5

În PASUL 5 (training):
1. Citim `tier5/*.tfrecord` (toate shard-urile)
2. Pentru tier_N (N ∈ {1..5}), filtrăm in-memory după `lesion_id` din `tier{N}_train.csv`
3. Aplicăm augmentare ONLINE (random crop 224, flip, rotation, etc.)
4. Antrenăm

Pentru val/test: citim TFRecord-urile direct, fără filtrare, fără augmentare (doar center crop la 224).